# Query Transformation (HyDE & Multi-Query): fix the question before you retrieve — from scratch

A user's **raw query is often a poor retrieval probe** — short, phrased as a *question*, and living
in a different neighbourhood of the embedding space than the *answer passages* it's looking for (the
**question ↔ document asymmetry**). This notebook builds the two fixes from primitives and **measures**
each on a **real** encoder:

- **HyDE** — write a hypothetical *answer*, retrieve with **its** embedding (it lives in answer-space).
- **Multi-Query** — fan the query into N reformulations, retrieve each, **fuse** the union with RRF.

> **Honesty (read first):** HyDE and Multi-Query both need a **generative LLM** to write the hypothetical
> / the reformulations. This environment is **encoder-only** (sentence-transformers, no generative model),
> so we do **not** fake an LLM. The **retrieval is 100% real and measured** — the same `all-MiniLM-L6-v2`
> bi-encoder chapters 3 & 5 use, over chapter 1's corpus — and the **generation step is shown with fixed,
> clearly-labelled exemplars** standing in for an LLM. Every number below is measured, asserted before it
> is claimed, and reproducible. Only the *text* of the hypotheticals/paraphrases is hand-authored.

It reuses the canonical functions in `query_transformation.py` (which import chapter 5's `DenseRetriever`
+ RRF + metrics, which import chapter 1's corpus) so the chapters share one source of truth.

## Step 1 — set up: import the canonical functions

Everything below uses the same functions the concept page and the `.py` module use — no logic is
re-defined in the notebook. We print the versions + device for reproducibility (the retrieval math is
pure numpy; the encoder runs on CPU).

In [1]:
import numpy as np

from query_transformation import (
    TOP_K,
    DenseRetriever,
    build_hyde_probes,
    build_multiquery_probe,
    cosine_to_gold,
    recall_at_k,
    reciprocal_rank,
    reciprocal_rank_fusion,
    retrieve_hyde,
    retrieve_multiquery,
    retrieve_raw,
    union_recall_independent,
)
from hybrid_search import full_corpus

print('numpy:', np.__version__)
try:
    import torch
    avail = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
    # the encoder is pinned to CPU (ch5's DenseRetriever) for determinism; we report the
    # available accelerator only for context — the demo runs the encoder on CPU regardless.
    print('torch:', torch.__version__, '| accelerator available:', avail, '| encoder runs on: cpu')
except ImportError:
    print('torch: not installed (retrieval is pure numpy — unaffected)')

corpus = full_corpus()
dense = DenseRetriever(corpus)
print(f'corpus: {len(corpus)} passages | dense lens: {dense.backend}')

numpy: 2.4.6


torch: 2.12.0 | accelerator available: mps | encoder runs on: cpu


corpus: 11 passages | dense lens: sentence-transformers/all-MiniLM-L6-v2


## Step 2 — the corpus and the probes

Chapter 1's eight Helios-7 passages plus chapter 5's three blind-spot lines (11 total). The probes are
hand-authored to expose the asymmetry: their gold answers are phrased *unlike* the questions that seek
them. Gold indices are resolved **by content**, so they stay correct if the corpus order changes.

In [2]:
for i, d in enumerate(corpus):
    print(f'  doc[{i:2d}]  {d}')

hyde_probes = build_hyde_probes(corpus)
mq_probe = build_multiquery_probe(corpus)
print()
for p in hyde_probes:
    print(f'HyDE probe [{p.label}]')
    print(f'    Q   : {p.query}')
    print(f'    gold: doc[{p.gold}] = {corpus[p.gold]}')
print(f'Multi-Query probe [{mq_probe.label}]')
print(f'    Q   : {mq_probe.query}')
print(f'    gold: doc[{mq_probe.gold}] = {corpus[mq_probe.gold]}')

  doc[ 0]  The Helios-7 satellite was launched on March 3rd, 2024 from the Kourou spaceport.
  doc[ 1]  Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.
  doc[ 2]  The project lead for Helios-7 is Dr. Amara Okoye, based in the Nairobi office.
  doc[ 3]  Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  doc[ 4]  Photosynthesis converts carbon dioxide and water into glucose using sunlight.
  doc[ 5]  The Eiffel Tower in Paris was completed in 1889 for the World's Fair.
  doc[ 6]  A standard chessboard has 64 squares arranged in an 8 by 8 grid.
  doc[ 7]  Water boils at 100 degrees Celsius at standard atmospheric pressure.
  doc[ 8]  Error E-4011 appeared in the Helios-7 telemetry stream.
  doc[ 9]  Climbing steadily, Helios-7 rose skyward moments past liftoff.
  doc[10]  The Helios-7 ground team spent the afternoon investigating several telemetry errors and console warnings.

HyDE probe [orbit (raw question mismatches the an

## Step 3 — feel the asymmetry: a question sits far from its own answer

The whole problem in one measurement. Embed the raw question and its gold answer, take the cosine, and
watch how much closer a **hypothetical answer** sits. `cosine_to_gold` uses the real MiniLM encoder
(unit-norm rows, so cosine = dot product — chapter 3's geometry).

In [3]:
print(f"{'probe':<12} | {'cos(question,gold)':>18} | {'cos(HyDE,gold)':>15} | {'lift':>7}")
print('-' * 62)
for p in hyde_probes:
    cos_q = cosine_to_gold(dense, corpus, p.query, p.gold)
    cos_h = cosine_to_gold(dense, corpus, p.hyde_good, p.gold)
    tag = p.label.split(' (')[0]
    print(f'{tag:<12} | {cos_q:>18.3f} | {cos_h:>15.3f} | {cos_h - cos_q:>+7.3f}')
    assert cos_h > cos_q, 'the hypothetical answer must embed closer to the gold than the question'
print('\n-> on every probe, the hypothetical ANSWER is closer to the gold than the raw question')

probe        | cos(question,gold) |  cos(HyDE,gold) |    lift
--------------------------------------------------------------
orbit        |              0.736 |           0.927 |  +0.191
exact-code   |              0.761 |           0.938 |  +0.177

-> on every probe, the hypothetical ANSWER is closer to the gold than the raw question


![The question≠answer gap, drawn in 2D (PCA of real all-MiniLM embeddings). The raw question (blue) sits
far from its gold answer (green star); the HyDE hypothetical (amber diamond) lands on top of the gold
cluster. Generated by `make_figures_07.py`.](../../images/rag07_asymmetry_2d.png)

## Step 4 — HyDE: retrieve with the hypothetical answer's embedding

The **only** change from ordinary retrieval is *what text we embed to search with*: the hypothetical
answer, not the question. Watch the exact-code probe's gold climb from rank #2 (raw) to #1 (HyDE).

In [4]:
def gold_rank(indices, gold):
    return f'#{list(indices).index(gold)+1}' if gold in indices else 'MISS'

for p in hyde_probes:
    raw = retrieve_raw(dense, p.query)
    hyde = retrieve_hyde(dense, p.hyde_good)
    print(f'PROBE [{p.label}]')
    print(f'  gold = doc[{p.gold}]: {corpus[p.gold]}')
    print(f'  RAW  top-{TOP_K} ids {list(raw)}   gold rank: {gold_rank(raw, p.gold)}')
    print(f'  HyDE top-{TOP_K} ids {list(hyde)}   gold rank: {gold_rank(hyde, p.gold)}\n')

PROBE [orbit (raw question mismatches the answer's vocabulary)]
  gold = doc[3]: Helios-7 completes one orbit of Earth every 97 minutes in a sun-synchronous orbit.
  RAW  top-3 ids [3, 1, 9]   gold rank: #1
  HyDE top-3 ids [3, 9, 0]   gold rank: #1

PROBE [exact-code (a chatty distractor out-embeds the terse answer)]
  gold = doc[8]: Error E-4011 appeared in the Helios-7 telemetry stream.
  RAW  top-3 ids [10, 8, 0]   gold rank: #2
  HyDE top-3 ids [8, 10, 0]   gold rank: #1



![The HyDE pipeline (query → hypothetical answer → embed the answer → retrieve) and the measured cosine
lift toward the gold (0.736→0.927, 0.761→0.938). Generated by `make_figures_07.py`.](../../images/rag07_hyde_pipeline.png)

**The intuition GIF — the HyDE jump.** The raw question sits far from its gold answer; the LLM rewrites
it as a hypothetical answer and the retrieval probe *travels* across the embedding space into the gold's
neighbourhood, and the gold's rank snaps #2 → #1. Endpoints are real embeddings; the travel is interpolated.

![Animated — the HyDE jump: the probe morphs from the question to the hypothetical answer and crosses the
embedding space into the gold cluster. Generated by `make_animation_07.py`.](../../images/rag07_hyde_jump.gif)

## Step 5 — the pitfall: a *wrong* hypothetical

The killer HyDE question: *what if the hypothetical is factually wrong?* We hand it a hypothetical that
is on-topic but **wrong on the specifics** (e.g. "error E-9999" instead of E-4011) and measure. Two
honest lessons: (a) a wrong hypothetical is **always worse** than a good one, and (b) it can even dip
just **below** the raw question's cosine — yet the gold **still ranks #1**, because retrieval acts on the
*ordering of the whole corpus*, not the gold's absolute cosine. The encoder's dense bottleneck keeps the
topic and drops the fabricated detail (HyDE's core claim).

In [5]:
for p in hyde_probes:
    cos_q = cosine_to_gold(dense, corpus, p.query, p.gold)
    cos_good = cosine_to_gold(dense, corpus, p.hyde_good, p.gold)
    cos_wrong = cosine_to_gold(dense, corpus, p.hyde_wrong, p.gold)
    wrong_rank = gold_rank(retrieve_hyde(dense, p.hyde_wrong), p.gold)
    tag = p.label.split(' (')[0]
    beats_q = 'yes' if cos_wrong > cos_q else 'no (dips below the question)'
    print(f'PROBE [{tag}]')
    print(f'  cos(question,gold)   = {cos_q:.3f}')
    print(f'  cos(HyDE-good,gold)  = {cos_good:.3f}')
    print(f'  cos(HyDE-WRONG,gold) = {cos_wrong:.3f}   (wrong facts, same topic)')
    print(f'  wrong cosine still > question\'s? {beats_q}')
    print(f'  HyDE-WRONG gold rank = {wrong_rank}   <- the number that actually matters\n')
    assert cos_wrong < cos_good, 'a wrong hypothetical should be WORSE than a good one'
    assert wrong_rank == '#1', 'even a wrong hypothetical should still retrieve the gold at #1'
print('-> wrong details cost similarity (wrong < good, always) and can dip under the question, yet the')
print('   gold still ranks #1: the bottleneck keeps the topical neighbourhood. Rank, not cosine, is what')
print('   retrieval acts on.')

PROBE [orbit]
  cos(question,gold)   = 0.736
  cos(HyDE-good,gold)  = 0.927
  cos(HyDE-WRONG,gold) = 0.832   (wrong facts, same topic)
  wrong cosine still > question's? yes
  HyDE-WRONG gold rank = #1   <- the number that actually matters

PROBE [exact-code]
  cos(question,gold)   = 0.761
  cos(HyDE-good,gold)  = 0.938
  cos(HyDE-WRONG,gold) = 0.750   (wrong facts, same topic)
  wrong cosine still > question's? no (dips below the question)
  HyDE-WRONG gold rank = #1   <- the number that actually matters

-> wrong details cost similarity (wrong < good, always) and can dip under the question, yet the
   gold still ranks #1: the bottleneck keeps the topical neighbourhood. Rank, not cosine, is what
   retrieval acts on.


## Step 6 — Multi-Query: N reformulations, retrieve each, fuse the union with RRF

A deliberately **vague** query ("What's the deal with Helios-7's imaging?") is a weak probe. We expand it
into three reformulations (an LLM would write these; here they're fixed exemplars), retrieve each, and
fuse the ranked lists with **RRF** (chapter 5's rank fusion — no score normalization, high-in-any-list
wins). Each reformulation ranks the gold #1; RRF locks it at #1 overall.

In [6]:
print(f'under-specified query: {mq_probe.query}')
print(f'gold = doc[{mq_probe.gold}]: {corpus[mq_probe.gold]}\n')
raw = retrieve_raw(dense, mq_probe.query)
print(f'RAW query        top-{TOP_K} ids {list(raw)}   gold rank: {gold_rank(raw, mq_probe.gold)}')
for i, para in enumerate(mq_probe.paraphrases, start=1):
    res = dense.search(para, k=TOP_K).indices
    print(f'  reformulation {i}: {para}')
    print(f'    top-{TOP_K} ids {list(res)}   gold rank: {gold_rank(res, mq_probe.gold)}   hit@{TOP_K}: {recall_at_k(res, mq_probe.gold):.0f}')
all_queries = (mq_probe.query, *mq_probe.paraphrases)
fused = retrieve_multiquery(dense, all_queries)
print(f'\nMULTI-QUERY (fused) top-{TOP_K} ids {list(fused)}   gold rank: {gold_rank(fused, mq_probe.gold)}')
assert recall_at_k(fused, mq_probe.gold) >= recall_at_k(raw, mq_probe.gold), 'fusing must not lose the gold'

# The exact RRF FUSED SCORES the figure/prose quote — computed with the SAME canonical
# reciprocal_rank_fusion the pipeline uses, over the 3 reformulations' FULL-CORPUS ranks.
para_lists = [dense.all_scores(q) for q in mq_probe.paraphrases]
rrf_full = reciprocal_rank_fusion(para_lists, k_rrf=60, k=len(corpus))  # full ranked list + scores
rrf_score = dict(zip(rrf_full.indices, rrf_full.scores))
print('\nRRF fused scores over the 3 reformulations (full-corpus ranks, top-5):')
for r, d in enumerate(rrf_full.indices[:5], start=1):
    tag = '  <- gold' if d == mq_probe.gold else ''
    print(f'  #{r} doc[{d}]  RRF = {rrf_score[d]:.4f}{tag}')
print(f'\nRRF(doc[{mq_probe.gold}]) = {rrf_score[mq_probe.gold]:.4f}   '
      f"(the exact number the page + figure quote; = 3 x 1/(60+1) since the gold is #1 in all three)")
assert abs(rrf_score[mq_probe.gold] - 0.0492) < 1e-4, 'gold RRF score must equal the quoted 0.0492'

# The FULL pipeline (retrieve_multiquery) fuses the ORIGINAL query alongside the 3 reformulations,
# so the gold's fused score is a touch higher (still #1) -- the other number the page quotes.
full_lists = [dense.all_scores(q) for q in all_queries]  # original query + 3 reformulations
rrf_pipeline = reciprocal_rank_fusion(full_lists, k_rrf=60, k=len(corpus))
pipeline_score = dict(zip(rrf_pipeline.indices, rrf_pipeline.scores))[mq_probe.gold]
print(f'RRF(doc[{mq_probe.gold}]) over query + 3 reformulations (full pipeline) = {pipeline_score:.4f}  (still #1)')
assert abs(pipeline_score - 0.0653) < 1e-4, 'full-pipeline gold RRF score must equal the quoted 0.0653'

under-specified query: What's the deal with Helios-7's imaging?
gold = doc[1]: Helios-7 carries a hyperspectral imager with a ground resolution of 4 meters.

RAW query        top-3 ids [10, 1, 2]   gold rank: #2
  reformulation 1: What is the ground resolution of the Helios-7 imager?
    top-3 ids [1, 0, 2]   gold rank: #1   hit@3: 1
  reformulation 2: What type of imaging sensor does Helios-7 carry?
    top-3 ids [1, 2, 10]   gold rank: #1   hit@3: 1
  reformulation 3: How detailed are the images Helios-7 captures of the ground?
    top-3 ids [1, 9, 10]   gold rank: #1   hit@3: 1



MULTI-QUERY (fused) top-3 ids [1, 10, 2]   gold rank: #1

RRF fused scores over the 3 reformulations (full-corpus ranks, top-5):
  #1 doc[1]  RRF = 0.0492  <- gold
  #2 doc[2]  RRF = 0.0476
  #3 doc[10]  RRF = 0.0474
  #4 doc[0]  RRF = 0.0471
  #5 doc[9]  RRF = 0.0464

RRF(doc[1]) = 0.0492   (the exact number the page + figure quote; = 3 x 1/(60+1) since the gold is #1 in all three)


RRF(doc[1]) over query + 3 reformulations (full pipeline) = 0.0653  (still #1)


![RRF over the chapter's three reformulations: the 1/(k+rank) weight curve and a worked fusion of the
reformulations' full-corpus rankings (top-5 shown); the gold doc[1] is #1 in every list, so RRF fuses
it to #1 with score 0.0492 — the exact number printed in the cell above. Generated by
`make_figures_07.py`.](../../images/rag07_rrf_fusion.png)

## Step 7 — the union-recall law: 1 − ∏(1 − pᵢ), and why independence is optimistic

If reformulation *i* retrieves the gold with probability *pᵢ*, the **union** hits with probability
1 − ∏(1 − pᵢ) — climbing toward 1 as N grows. That assumes **independence**, which is optimistic:
paraphrases share meaning, so their misses correlate and real recall lags the ceiling. On this tiny
corpus each reformulation already hits (p̂ = 1.0 → union = 1.0, recall saturated), so we also show the
**law** for a weaker per-query p to make the shape visible — labelled illustrative where it is.

In [7]:
# measured on this corpus: each reformulation's hit@k, and the resulting union
per_query_hits = [recall_at_k(dense.search(q, k=TOP_K).indices, mq_probe.gold) for q in mq_probe.paraphrases]
p_hat = float(np.mean(per_query_hits))
print(f'measured per-reformulation hit@{TOP_K}: {per_query_hits}  ->  p_hat = {p_hat:.2f}')
print(f'measured union recall (all reformulations): {recall_at_k(fused, mq_probe.gold):.2f}  (saturated on 11 docs)\n')

# the LAW on a harder corpus (illustrative per-query p), to show the 1-(1-p)^N climb
print('the union-recall law 1-(1-p)^N (illustrative p to show the shape):')
print(f"{'N':>3} | {'p=0.40':>8} | {'p=0.55':>8} | {'p=0.70':>8}")
print('-' * 34)
for n in (1, 2, 3, 5, 8):
    row = [union_recall_independent((p,) * n) for p in (0.40, 0.55, 0.70)]
    print(f'{n:>3} | {row[0]:>8.3f} | {row[1]:>8.3f} | {row[2]:>8.3f}')
assert union_recall_independent((0.55, 0.55, 0.55)) > 0.55, 'the union must exceed a single query'

measured per-reformulation hit@3: [1.0, 1.0, 1.0]  ->  p_hat = 1.00
measured union recall (all reformulations): 1.00  (saturated on 11 docs)

the union-recall law 1-(1-p)^N (illustrative p to show the shape):
  N |   p=0.40 |   p=0.55 |   p=0.70
----------------------------------
  1 |    0.400 |    0.550 |    0.700
  2 |    0.640 |    0.798 |    0.910
  3 |    0.784 |    0.909 |    0.973
  5 |    0.922 |    0.982 |    0.998
  8 |    0.983 |    0.998 |    1.000


![Multi-Query union recall vs N: the independent ceiling 1−(1−p)^N vs a correlated curve that lags it
(the shaded band is the cost of correlation). p and ρ are illustrative to show the law; the measured
corpus saturates. Generated by `make_figures_07.py`.](../../images/rag07_union_recall.png)

## Step 8 — the payoff: MRR raw vs transformed

Put it together. Over each transform's probe set, MRR before vs after. recall@3 is 1.0 everywhere on
this 11-doc corpus (top-3 of 11 almost always contains the gold — **saturated**), so the **MRR** carries
the signal: it sees the rank improvement (gold #2 → #1) that recall@3 is too coarse to catch.

In [8]:
raw_rr = [reciprocal_rank(retrieve_raw(dense, p.query), p.gold) for p in hyde_probes]
hyde_rr = [reciprocal_rank(retrieve_hyde(dense, p.hyde_good), p.gold) for p in hyde_probes]
raw_rec = [recall_at_k(retrieve_raw(dense, p.query), p.gold) for p in hyde_probes]
hyde_rec = [recall_at_k(retrieve_hyde(dense, p.hyde_good), p.gold) for p in hyde_probes]
mq_raw_rr = reciprocal_rank(retrieve_raw(dense, mq_probe.query), mq_probe.gold)
mq_fused_rr = reciprocal_rank(fused, mq_probe.gold)
mq_raw_rec = recall_at_k(retrieve_raw(dense, mq_probe.query), mq_probe.gold)
mq_fused_rec = recall_at_k(fused, mq_probe.gold)

print(f"{'method':<28} | {'MRR':>6} | {'recall@'+str(TOP_K):>9}")
print('-' * 50)
print(f"{'raw query (HyDE set)':<28} | {np.mean(raw_rr):>6.3f} | {np.mean(raw_rec):>9.3f}")
print(f"{'HyDE (HyDE set)':<28} | {np.mean(hyde_rr):>6.3f} | {np.mean(hyde_rec):>9.3f}")
print(f"{'raw query (MQ probe)':<28} | {mq_raw_rr:>6.3f} | {mq_raw_rec:>9.3f}")
print(f"{'Multi-Query (MQ probe)':<28} | {mq_fused_rr:>6.3f} | {mq_fused_rec:>9.3f}")

assert np.mean(hyde_rr) > np.mean(raw_rr), 'HyDE must lift MRR over the raw query'
assert mq_fused_rr >= mq_raw_rr, 'Multi-Query must not reduce MRR'
print('\nHyDE lifts MRR over the raw query; Multi-Query lifts (or holds) recall+MRR: True')

method                       |    MRR |  recall@3
--------------------------------------------------
raw query (HyDE set)         |  0.750 |     1.000
HyDE (HyDE set)              |  1.000 |     1.000
raw query (MQ probe)         |  0.500 |     1.000
Multi-Query (MQ probe)       |  1.000 |     1.000

HyDE lifts MRR over the raw query; Multi-Query lifts (or holds) recall+MRR: True


![The payoff: MRR raw vs transformed on each probe set (measured). HyDE 0.750→1.000; Multi-Query
0.500→1.000. recall@3 saturated at 1.00, so MRR carries the signal. Generated by
`make_figures_07.py`.](../../images/rag07_before_after.png)

## Step 9 — when to SKIP it: an already-precise query

Transformation exists to fix *weak* probes. A query that's already an excellent probe — a keyword/exact
query the raw retriever nails at #1 — has nothing to fix, so a HyDE detour can only add latency and drift.

In [9]:
precise_query = 'Helios-7 hyperspectral imager ground resolution 4 meters'
precise_gold = next(i for i, d in enumerate(corpus) if 'hyperspectral' in d)
raw = retrieve_raw(dense, precise_query)
cos_precise = cosine_to_gold(dense, corpus, precise_query, precise_gold)
print(f'precise query: {precise_query}')
print(f'  RAW top-{TOP_K} ids {list(raw)}   gold rank: {gold_rank(raw, precise_gold)}')
print(f'  cos(precise query, gold) = {cos_precise:.3f}  -- already high; the raw probe is a great probe')
assert gold_rank(raw, precise_gold) == '#1', 'a precise query should already rank the gold #1'
print('  -> skip transformation here: it adds latency + drift risk, not recall')

precise query: Helios-7 hyperspectral imager ground resolution 4 meters
  RAW top-3 ids [1, 0, 9]   gold rank: #1
  cos(precise query, gold) = 0.932  -- already high; the raw probe is a great probe
  -> skip transformation here: it adds latency + drift risk, not recall


## Step 10 — the library one-liners (need a generative LLM)

In production you don't hand-roll either transform. Both **require a generative LLM**, so they won't run
in this encoder-only environment — shown for reference:

```python
# LlamaIndex — HyDE (needs Settings.llm)
from llama_index.core.indices.query.query_transform.base import HyDEQueryTransform
from llama_index.core.query_engine import TransformQueryEngine
hyde = HyDEQueryTransform(include_original=True)          # embed the hypothetical (+ optionally the original)
engine = TransformQueryEngine(index.as_query_engine(), query_transform=hyde)
response = engine.query('What telemetry error did Helios-7 report?')

# LangChain — Multi-Query (needs an LLM)
from langchain.retrievers.multi_query import MultiQueryRetriever
# default prompt -> 3 reformulations; returns the UNIQUE UNION of retrieved docs
retriever = MultiQueryRetriever.from_llm(retriever=base_retriever, llm=chat_llm)
docs = retriever.invoke("What's the deal with Helios-7's imaging?")
```

Both hide exactly the transform-then-retrieve(-then-fuse) pipeline this notebook built by hand.

---

### Recap

- A raw question embeds in **question-space**, far from its **answer passage** (the asymmetry).
- **HyDE** retrieves with a hypothetical *answer*'s embedding; the encoder's bottleneck forgives wrong facts.
- **Multi-Query** fans into N reformulations and fuses the union (RRF); union recall 1−∏(1−pᵢ), optimistic.
- **Skip** it for precise/keyword queries; reach for it on vague, vocabulary-mismatched ones.
- Full write-up: [the concept page](../07-Query-Transformation-HyDE-Multi-Query.md).